In [1]:
from IPython.display import Image

- https://verl.readthedocs.io/en/latest/hybrid_flow.html
- https://verl.readthedocs.io/en/latest/advance/agent_loop.html
- https://verl.readthedocs.io/en/latest/start/agentic_rl.html

### architect

In [4]:
Image(url='../imgs/verl-dataflow-constraint.png', width=600)

- 通过精心设计模型的**放置（Placement）策略，将一个逻辑上的数据流图（Dataflow Graph）**映射到物理的计算集群上，从而在满足数据依赖约束的同时，最大化并行度以提升效率。
    - Actor: 0-1, Critic: 2-3, Ref/RM: 4-5 (colocated, 共同部署)

### vllm wakeup & sleep mode

- By default, verl requires vLLM to enable **sleep mode**, which allows vLLM to offload **GPU memory to CPU memory** after rollout. However, this feature is still under review by the vLLM community.
- 在不需要进行AI推理时，可以将模型从昂贵的GPU显存中“请出去”（sleep），释放资源给其他任务；当需要再次推理时，可以快速将其“唤醒”并重新加载回GPU（wake_up），而无需完全重新创建和初始化模型。

```python
import torch
import psutil
from vllm import LLM

llm = LLM(model="Qwen/Qwen2.5-3B-Instruct", enable_sleep_mode=True)

def run_inference(prompt):
        outputs = llm.generate(prompt)
        for output in outputs:
                prompt = output.prompt
                generated_text = output.outputs[0].text
                print(f"Prompt: {prompt!r}, Generated text: {generated_text!r}")


def print_memory_usage():
    # CPU Memory
    process = psutil.Process()
    cpu_mem_used = process.memory_info().rss / 1024**3  # in GB
    print(f"Used CPU memory (by this process): {cpu_mem_used:.2f} GB")

    # GPU Memory
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        used = (total - free) / 1024**3
        total = total / 1024**3
        print(f"Used GPU memory: {used:.2f} GB, Total GPU memory: {total:.2f} GB")
    else:
        print("CUDA is not available.")


print("Running first inference to load model to GPU...")
run_inference("San Francisco is")

print("\nMemory Usage (after first inference, model is loaded):")
torch.cuda.empty_cache()
print_memory_usage()

print("\nPutting model to sleep...")
llm.sleep()

print("\nMemory Usage (after sleep):")
torch.cuda.empty_cache()
print_memory_usage()

print("\nWaking up model...")
llm.wake_up()

print("\nMemory Usage (after wakeup):")
torch.cuda.empty_cache()
print_memory_usage()

print("\nRunning second inference to verify...")
run_inference("Paris is")
```

```
Running first inference to load model to GPU...
Prompt: 'San Francisco is', Generated text: ' a city that has been at the forefront of many progressive social movements. The '

Memory Usage (after first inference, model is loaded):
Used CPU memory (by this process): 0.75 GB
Used GPU memory: 73.78 GB, Total GPU memory: 79.33 GB

Putting model to sleep...

Memory Usage (after sleep):
Used CPU memory (by this process): 0.84 GB
Used GPU memory: 2.82 GB, Total GPU memory: 79.33 GB


Waking up model...

Memory Usage (after wakeup):
Used CPU memory (by this process): 0.84 GB
Used GPU memory: 68.18 GB, Total GPU memory: 79.33 GB

Running second inference to verify...

Prompt: 'Paris is', Generated text: ' the capital and most populous city of France. It has been an important political,'
```

- `torch.cuda.memory_allocated()` 为什么不行？因为它只追踪由PyTorch自身内存分配器所分配的显存。vLLM为了极致的性能，使用了自己的内存管理系统（例如 PagedAttention），直接通过底层的CUDA API来管理显存，从而绕过了PyTorch的分配器。因此，PyTorch的函数看不到vLLM占用的显存。
- `torch.cuda.mem_get_info()` 为什么可以？因为它直接从NVIDIA驱动层面查询GPU的状态，返回全局的“总显存”和“空闲显存”。通过 总显存 - 空闲显存，我们就能计算出当前GPU上所有进程占用的总显存，这自然也包括了vLLM所占用的部分。
- sleep() 并不将模型权重缓存到CPU内存。它通过调用资源管理器（sharding_manager）的退出方法 (`__exit__`) 来释放GPU上的资源。
wake_up() 也不从CPU内存中恢复模型。它通过调用资源管理器的进入方法 (`__enter__`)，触发一个从磁盘重新加载模型到GPU的过程。

```python
def load_model(self, *args, **kwargs):
    self.inference_engine.load_model(*args, **kwargs)

    # inference engine is initialized now, update sharding manager
    self.sharding_manager.inference_engine = self.inference_engine
    self.sharding_manager.model_runner = self.inference_engine.worker.model_runner

    _monkey_patch_compute_logits(self.inference_engine.worker.model_runner.model, len(self.tokenizer))

def sleep(self, *args, **kwargs):
    """Offload model weights and discard kv cache."""
    if self.is_sleep:
        return
    self.sharding_manager.__exit__(None, None, None)
    self.is_sleep = True

def wake_up(self, *args, **kwargs):
    """Load model weights and build kv cache."""
    if not self.is_sleep:
        return
    self.sharding_manager.__enter__()  # pylint: disable=C2801
    self.is_sleep = False
```

- 你的服务器上只有一张GPU，但你需要运行多个不同的AI模型（比如一个大语言模型、一个图像生成模型）。当语言模型空闲时，你可以让它 sleep()，释放出宝贵的VRAM给图像模型使用。当需要处理语言任务时，再 wake_up() 语言模型。这避免了完全销毁和重新加载模型的巨大开销，实现了在单一GPU上对多种服务的动态资源调度，最大化硬件利用率。


### agent loop